In [1]:
import os
import warnings
import time

import dask.dataframe as dd
import pyarrow.parquet as pq
import pyarrow as pa
from fastparquet import write

import pandas as pd
import numpy as np
import math

import pickle

from matplotlib import pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
import plotly.graph_objects as go

import lifelines
from lifelines.utils import concordance_index
from lifelines.statistics import logrank_test
from sksurv.util import Surv
from sksurv.metrics import concordance_index_ipcw

import tensorflow as tf
import tensorflow_probability as tfp

config = tf.compat.v1.ConfigProto()
config.gpu_options.allow_growth = True
sess = tf.compat.v1.Session(config = config)

from tensorflow import keras
from tensorflow.keras import optimizers, initializers, regularizers, layers

import scipy.stats as stats
from scipy.stats import norm, t, probplot, pearsonr, spearmanr, rankdata
from scipy.stats import truncnorm as truncnorm_scipy
from scipy.stats import gamma as gamma_dist
from scipy.special import gamma

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, KFold

# import thetaflow as thf
import modelnn2 as thf

import json
import gc
import glob
from pathlib import Path

2026-05-28 16:30:35.670085: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1779996635.776894    6627 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779996635.806802    6627 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1779996636.047730    6627 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779996636.047766    6627 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779996636.047768    6627 computation_placer.cc:177] computation placer alr

In [2]:
# Função de Parsing estritamente executada no grafo interno em C++
def parse_tfrecord_fn(serialized_example):
    # Declaramos o mapa estruturado com os tamanhos exatos fixados
    feature_description = {
        'features': tf.io.FixedLenFeature([n_features], tf.float32),
        'time': tf.io.FixedLenFeature([], tf.float32),
        'event': tf.io.FixedLenFeature([], tf.float32),
    }
    
    # Decodifica a string binária em tensores puros
    example = tf.io.parse_single_example(serialized_example, feature_description)
    
    # Retorna EXATAMENTE a tupla plana com 3 elementos esperada pelo seu loop customizado
    return example['features'], example['time'], example['event']

In [3]:
# Definições Iniciais Sincronizadas
train_data_dir = "tfrecords_dir_train/shard_*.tfrecord"
files = glob.glob(train_data_dir)

# Suas 102 colunas do parquet original menos as 2 colunas de destino excluídas pelo drop
n_features = 100 
batch_size = 20000

# 3. Criando o fluxo de leitura paralela concorrente
train_dataset_tfrecord = tf.data.Dataset.list_files(files, shuffle=True, seed=10)

# Mágica do I/O: lê múltiplos arquivos ao mesmo tempo, intercalando os bytes em threads de segundo plano
train_dataset_tfrecord = train_dataset_tfrecord.interleave(
    lambda filename: tf.data.TFRecordDataset(filename, num_parallel_reads=tf.data.AUTOTUNE),
    cycle_length=tf.data.AUTOTUNE,
    num_parallel_calls=tf.data.AUTOTUNE
)

# 4. Otimização final do Grafo de Fluxo NATIVO
train_dataset_tfrecord = (
    train_dataset_tfrecord
    .map(parse_tfrecord_fn, num_parallel_calls=tf.data.AUTOTUNE) # Parsing paralelizado pura em C++
    # .shuffle(buffer_size=40000, reshuffle_each_iteration=True, seed=10)
    .batch(batch_size)
    .prefetch(tf.data.AUTOTUNE) # CPU prepara o batch t+1 na memória enquanto a GPU calcula o batch t
)

In [4]:
# Definições Iniciais Sincronizadas
test_data_dir = "tfrecords_dir_test/shard_*.tfrecord"
files = glob.glob(train_data_dir)

# Suas 102 colunas do parquet original menos as 2 colunas de destino excluídas pelo drop
n_features = 100 
batch_size = 20000

# 3. Criando o fluxo de leitura paralela concorrente
test_dataset_tfrecord = tf.data.Dataset.list_files(files, shuffle=True, seed=10)

# Mágica do I/O: lê múltiplos arquivos ao mesmo tempo, intercalando os bytes em threads de segundo plano
test_dataset_tfrecord = test_dataset_tfrecord.interleave(
    lambda filename: tf.data.TFRecordDataset(filename, num_parallel_reads=tf.data.AUTOTUNE),
    cycle_length=tf.data.AUTOTUNE,
    num_parallel_calls=tf.data.AUTOTUNE
)

# 4. Otimização final do Grafo de Fluxo NATIVO
test_dataset_tfrecord = (
    test_dataset_tfrecord
    .map(parse_tfrecord_fn, num_parallel_calls=tf.data.AUTOTUNE) # Parsing paralelizado pura em C++
    # .shuffle(buffer_size=40000, reshuffle_each_iteration=True, seed=10)
    .batch(batch_size)
    .prefetch(tf.data.AUTOTUNE) # CPU prepara o batch t+1 na memória enquanto a GPU calcula o batch t
)

In [5]:
def survival_data_loader_generator(file_path, time_col, event_col):
    '''
        Lê um arquivo parquet em batches e cospe os tensores para o modelo.
    '''
    
    parquet_file = pq.ParquetFile(file_path)
    num_groups = parquet_file.num_row_groups
    # Itera sobre os blocos nativos do Parquet
    for i in range(num_groups):
        # Lê o i-ésimo grupo de observações separado no arquivo parquet
        table = parquet_file.read_row_group(i)
        
        # Converte apenas este batch para um DataFrame Pandas/Numpy
        df_batch = table.to_pandas()
        
        # Variáveis de Sobrevivência (Y)
        time = df_batch[time_col].values.astype(np.float32)
        event = df_batch[event_col].values.astype(np.float32)

        # Remove as colunas de tempo e censura, mantendo apenas as features
        df_batch = df_batch.drop(columns = [time_col, event_col])
        # Matriz de Features (X)
        X = df_batch.values.astype(np.float32)
        
        # As variáveis resposta são retornadas como uma lista
        # y = (time, event)
        
        # Entrega o batch pronto para o TensorFlow
        yield X, time, event

In [6]:
def survival_data_loader_generator(file_path, time_col, event_col):
    parquet_file = pq.ParquetFile(file_path)
    num_groups = parquet_file.num_row_groups
    
    for i in range(num_groups):
        table = parquet_file.read_row_group(i)
        
        # 1. Extração direta para NumPy (Muito mais rápido que .to_pandas())
        # pyarrow extrai a coluna e converte direto pro C-backend do numpy
        time = table.column(time_col).to_numpy().astype(np.float32)
        event = table.column(event_col).to_numpy().astype(np.float32)
        
        # 2. Dropamos as colunas alvo da tabela do pyarrow
        feature_cols = [c for c in table.column_names if c not in [time_col, event_col]]
        table_features = table.select(feature_cols)
        
        # 3. Conversão para NumPy 2D agilizada
        # O to_pandas() aqui é inevitável para colar as colunas colunares em matriz 2D,
        # MAS, como os dados já estão limpos e numéricos, fazemos de forma mais enxuta
        X = table_features.to_pandas().values.astype(np.float32)
        
        yield X, time, event

In [7]:
train_data_path = "train_data.parquet"
n_features = len(dd.read_parquet(train_data_path).columns)
time_col = "tempo"
event_col = "delta"
batch_size = 20000

# Criando o Dataset base com a nova Assinatura
train_dataset = tf.data.Dataset.from_generator(
    lambda: survival_data_loader_generator(train_data_path, time_col, event_col),
    
    output_signature=(
        # 1. Matriz X (Features)
        tf.TensorSpec(shape=(None, n_features-2), dtype=tf.float32),
        # 2. Vetor time (Tempo)
        tf.TensorSpec(shape=(None,), dtype=tf.float32), 
        # 3. Vetor event (Evento/Censura)
        tf.TensorSpec(shape=(None,), dtype=tf.float32)  
    )
)

# Substitua pelo número exato de observações que você obteve lendo o Parquet
total_observations = 11263238
# total_observations = 20001
batch_size = 20000

# Calcula o número total de passos (batches) por época
num_batches = math.ceil(total_observations / batch_size)

# Otimização do pipeline
train_dataset = (
    train_dataset
    .unbatch()                                                                 # Quebra os 512 blocos gigantes em linhas individuais fluidas
    # .shuffle(buffer_size = 40000, reshuffle_each_iteration = True, seed = 10)  # Embaralha as observações de cada batch
    .batch(batch_size)                                                         # Reagrupa em lotes exatos de batch_size para estabilizar o gradiente da Rede Neural
    # .prefetch(tf.data.AUTOTUNE)                                                # Manda a CPU preparar o próximo batch enquanto a GPU treina o atual
    .prefetch(2)                                                # Manda a CPU preparar o próximo batch enquanto a GPU treina o atual
    .apply(tf.data.experimental.assert_cardinality(num_batches))
)

In [8]:
# for i, batch_full_data in enumerate(train_dataset):
#     X, y, delta = batch_full_data
#     if((i+1) > 560 or (i+1) % 200 == 0):
#         print(i+1, X.shape)
#         print(X[:3,:5])

In [9]:
test_data_path = "test_data.parquet"
n_features = len(dd.read_parquet(test_data_path).columns)
time_col = "tempo"
event_col = "delta"
batch_size = 20000

# Criando o Dataset base com a nova Assinatura
test_dataset = tf.data.Dataset.from_generator(
    lambda: survival_data_loader_generator(test_data_path, time_col, event_col),
    
    output_signature=(
        # 1. Matriz X (Features)
        tf.TensorSpec(shape=(None, n_features-2), dtype=tf.float32),
        # 2. Vetor time (Tempo)
        tf.TensorSpec(shape=(None,), dtype=tf.float32), 
        # 3. Vetor event (Evento/Censura)
        tf.TensorSpec(shape=(None,), dtype=tf.float32)  
    )
)

# Substitua pelo número exato de observações que você obteve lendo o Parquet
total_observations = 4827103  
batch_size = 20000

# Calcula o número total de passos (batches) por época
num_batches = math.ceil(total_observations / batch_size)

# Otimização do pipeline
test_dataset = (
    test_dataset
    .unbatch()                                                                 # Quebra os 512 blocos gigantes em linhas individuais fluidas
    # .shuffle(buffer_size = 40000, reshuffle_each_iteration = True, seed = 10)  # Embaralha as observações de cada batch
    .batch(batch_size)                                                         # Reagrupa em lotes exatos de batch_size para estabilizar o gradiente da Rede Neural
    # .prefetch(tf.data.AUTOTUNE)                                                # Manda a CPU preparar o próximo batch enquanto a GPU treina o atual
    .prefetch(2)                                                # Manda a CPU preparar o próximo batch enquanto a GPU treina o atual
    .apply(tf.data.experimental.assert_cardinality(num_batches))
)

# Models

In [10]:
def build_weibull_model( ridge_penalty = 0.01, lasso_penalty = 0.01 ):
    # Shape parameter: constant for all patients
    # Scale parameter: modeled as a neural network output
    weibull_parameters = {
        "k": {"link": tf.math.exp, "link_inv": tf.math.log, "par_type": "independent", "shape": 1, "init": 1.0, "warmup_time": 0},
        "lam": {"link": tf.math.exp, "link_inv": tf.math.log, "par_type": "nn", "shape": 1, "init": 1.0, "warmup_time": 0}
    }

    def loglikelihood_loss(model, nn_output, data):
        # Unpack your data tuple
        X, y, delta = data
        
        k = model.get_variable("k")
        # k = model.get_variable("k", nn_output)
        log_lam = model.get_variable("lam", nn_output, get_raw_value = True)

        log_y = tf.math.log(y + 1.0e-7)
        log_hazard = tf.math.log(k) - k * log_lam + (k - 1.0) * log_y
        survival_term = tf.math.exp(k * (log_y - log_lam))

        loglik_terms = (delta * log_hazard) - survival_term
        neg_loglik = -tf.reduce_sum(loglik_terms)
        
        return neg_loglik

    def neural_network(model, seed = None):
        initializer = initializers.GlorotNormal(seed = seed)
        elastic_net = tf.keras.regularizers.L1L2(l1 = lasso_penalty, l2 = ridge_penalty)
        model.dense1 = layers.Dense(
            units = 32,
            activation = "gelu",
            kernel_initializer = initializer,
            kernel_regularizer = elastic_net,
            use_bias = True,
            dtype = tf.float32, 
            name = "gene_bottleneck"
        )
        model.dense2 = layers.Dense(
            units = 16,
            activation = "gelu",
            kernel_initializer = initializer,
            use_bias = True,
            dtype = tf.float32,
            name = "hidden_layer1"
        )
        model.dense3 = layers.Dense(
            units = 2,
            activation = "gelu",
            kernel_initializer = initializer,
            use_bias = True,
            dtype = tf.float32,
            name = "statistical_layer"
        )
        model.output_layer = layers.Dense(
            units = 1,
            activation = None,
            use_bias = True,
            kernel_initializer = initializer,
            dtype = tf.float32,
            name = "log_lambda"
        )
    
    def neural_network_call(model, x_input, training = False):
        x = model.dense1(x_input)
        x = model.dense2(x)
        x = model.dense3(x)
        x = model.output_layer(x)
        return x
    
    def neural_network_call_nolast(model, x_input):
        x = model.dense1(x_input)
        x = model.dense2(x)
        x = model.dense3(x)
        return x

    return weibull_parameters, loglikelihood_loss, neural_network, neural_network_call, neural_network_call_nolast

In [ ]:
with tf.device("/GPU:0"):
    weibull_parameters, weibull_loss, weibull_neural_network, weibull_call, weibull_call_nolast = \
    build_weibull_model( ridge_penalty = 0.01, lasso_penalty = 0.01 )
    seed = 10
    weibull_model = thf.ModelNN(weibull_parameters, weibull_loss,
                                weibull_neural_network, weibull_call,
                                weibull_call_nolast, input_dim = (100,), seed = seed)
    weibull_model.pre_train_model(epochs = None, x = train_dataset, data = None, n_train = 11263238, shuffle = True)
    weibull_model.train_model(epochs = 5000, x = train_dataset, data = None, n_train = 11263238,
                              shuffle = True,
                              get_covariances = True,
                              validation = True, val_prop = 0.2, force_training_validation = False,
                              optimizer_independent = optimizers.Adam(learning_rate = 0.01, clipnorm = 1.0),
                              optimizer_nn = optimizers.Adam(learning_rate = 0.01, clipnorm = 1.0),
                              fine_tune_nn_lr = 0.01, fine_tune_independent_lr = 0.01,
                              early_stopping = True, early_stopping_patience = 50, 
                              early_stopping_warmup = 10,
                              reduce_lr = True, reduce_lr_warmup = 0,
                              reduce_lr_factor = 0.5, reduce_lr_min_delta = 1.0e-3, reduce_lr_patience = 25,
                              reduce_lr_cooldown = 10, reduce_lr_min_lr = 1.0e-5,
                              fine_tune = True,
                              finetune_early_stopping = True, finetune_early_stopping_patience = 50,
                              finetune_early_stopping_warmup = 10,
                              finetune_reduce_lr = True, finetune_reduce_lr_warmup = 0,
                              finetune_reduce_lr_factor = 0.5, finetune_reduce_lr_min_delta = 1.0e-2, finetune_reduce_lr_patience = 25,
                              finetune_reduce_lr_cooldown = 10, finetune_reduce_lr_min_lr = 1.0e-5,
                              deterministic = True,
                              verbose = True, print_freq = 1,
                              train_batch_size = None, val_batch_size = None,
                              buffer_size = 4096, gradient_accumulation_steps = None,)

GPU detected. Activating GPU determinism. To reverse this, the Python environment (or kernel) must be restated.
Global seed set to 10.
Initializing training...
    Test final - Batches finished
epoch 0 batch 0
    Test 1 - Gradients done
    Test 2 - nan Gradients verification
    Test 3 - Gradients formatting
    Test 3 - Gradients cummulating
epoch 0 batch 1
    Test 1 - Gradients done
    Test 2 - nan Gradients verification
    Test 3 - Gradients formatting
    Test 3 - Gradients cummulating
epoch 0 batch 2
    Test 1 - Gradients done
    Test 2 - nan Gradients verification
    Test 3 - Gradients formatting
    Test 3 - Gradients cummulating
epoch 0 batch 3
    Test 1 - Gradients done
    Test 2 - nan Gradients verification
    Test 3 - Gradients formatting
    Test 3 - Gradients cummulating
epoch 0 batch 4
    Test 1 - Gradients done
    Test 2 - nan Gradients verification
    Test 3 - Gradients formatting
    Test 3 - Gradients cummulating
epoch 0 batch 5
    Test 1 - Gradients do

In [4]:
def converter_parquet_para_tfrecords(parquet_path, output_dir, num_shards=32):
    """
    Lê o arquivo Parquet pelos Row Groups nativos e converte em shards de TFRecord.
    """
    os.makedirs(output_dir, exist_ok=True)
    parquet_file = pq.ParquetFile(parquet_path)
    num_groups = parquet_file.num_row_groups
    
    # Nomes das colunas conforme seu dicionário atual
    time_col = "tempo"
    event_col = "delta"
    
    # Inicializa os writers binários para os shards (escrita balanceada)
    writers = [
        tf.io.TFRecordWriter(os.path.join(output_dir, f"shard_{i:03d}.tfrecord"))
        for i in range(num_shards)
    ]
    
    print(f"Iniciando conversão de {num_groups} Row Groups para {num_shards} TFRecords...")
    
    for i in range(num_groups):
        table = parquet_file.read_row_group(i)
        df_batch = table.to_pandas()
        
        # Separação dos alvos de sobrevivência
        times = df_batch[time_col].values.astype(np.float32)
        events = df_batch[event_col].values.astype(np.float32)
        
        # Separação e isolamento das exatas 100 features restantes
        df_features = df_batch.drop(columns=[time_col, event_col])
        features_matrix = df_features.values.astype(np.float32)
        
        # Iteração por linha dentro do Row Group para criar os buffers de protocolo
        for j in range(len(df_batch)):
            # Converte vetores e escalares em tipos reconhecidos pelo Protobuf do TF
            feature_list = tf.train.Feature(float_list=tf.train.FloatList(value=features_matrix[j]))
            time_feature = tf.train.Feature(float_list=tf.train.FloatList(value=[times[j]]))
            event_feature = tf.train.Feature(float_list=tf.train.FloatList(value=[events[j]]))
            
            # Monta o dicionário estruturado do registro
            example = tf.train.Example(features=tf.train.Features(feature={
                'features': feature_list,
                'time': time_feature,
                'event': event_feature
            }))
            
            # Distribui as linhas entre os shards via técnica round-robin (j % num_shards)
            writers[j % num_shards].write(example.SerializeToString())
            
        if (i + 1) % 50 == 0 or i == num_groups - 1:
            print(f"Progresso: {i + 1}/{num_groups} Row Groups serializados com sucesso.")
            
    # Fecha todos os arquivos abertos de escrita
    for w in writers:
        w.close()
    print("Conversão de dados binários concluída!")

# Execução única do pipeline de dados
converter_parquet_para_tfrecords("train_data.parquet", "tfrecords_dir_train", num_shards = 32)

Iniciando conversão de 512 Row Groups para 32 TFRecords...
Progresso: 50/512 Row Groups serializados com sucesso.
Progresso: 100/512 Row Groups serializados com sucesso.
Progresso: 150/512 Row Groups serializados com sucesso.
Progresso: 200/512 Row Groups serializados com sucesso.
Progresso: 250/512 Row Groups serializados com sucesso.
Progresso: 300/512 Row Groups serializados com sucesso.
Progresso: 350/512 Row Groups serializados com sucesso.
Progresso: 400/512 Row Groups serializados com sucesso.
Progresso: 450/512 Row Groups serializados com sucesso.
Progresso: 500/512 Row Groups serializados com sucesso.
Progresso: 512/512 Row Groups serializados com sucesso.
Conversão de dados binários concluída!


In [6]:
converter_parquet_para_tfrecords("test_data.parquet", "tfrecords_dir_test", num_shards = 16)

Iniciando conversão de 256 Row Groups para 16 TFRecords...
Progresso: 50/256 Row Groups serializados com sucesso.
Progresso: 100/256 Row Groups serializados com sucesso.
Progresso: 150/256 Row Groups serializados com sucesso.
Progresso: 200/256 Row Groups serializados com sucesso.
Progresso: 250/256 Row Groups serializados com sucesso.
Progresso: 256/256 Row Groups serializados com sucesso.
Conversão de dados binários concluída!
